# California Housing Price Prediction: Linear Regression Model
This notebook demonstrates building and evaluating a **Linear Regression model** to predict house prices using the California Housing dataset. 

## Objective
Predict the `median_house_value` using numerical features like location, housing age, rooms, population, and income, along with the categorical feature `ocean_proximity`.

## 1. Environment Setup
We start by importing all the necessary Python libraries for data handling, preprocessing, model training, evaluation, and visualization.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configure plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Environment setup complete!")

## 2. Data Loading
We load the California Housing dataset directly from the local `housing.csv` file and examine its structure.

In [ ]:
data_path = "housing.csv"
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
df.info()

## 3. Exploratory Data Analysis (EDA)
We look for missing values, analyze summary statistics, and visualize target distribution and feature correlations.

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values Per Column:")
print(missing_values[missing_values > 0])

In [ ]:
# Descriptive Statistics
df.describe()

In [ ]:
# Visualize distribution of the target variable (median_house_value)
plt.figure(figsize=(10, 6))
sns.histplot(df["median_house_value"], kde=True, color="#1a365d", bins=50)
plt.title("Distribution of Median House Value", fontsize=15, pad=15, color="#1a365d", weight="bold")
plt.xlabel("Median House Value ($)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Analyze feature correlations using a heatmap
plt.figure(figsize=(12, 10))
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numerical_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True, 
    cmap="coolwarm", 
    fmt=".2f", 
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Numerical Features Correlation Heatmap", fontsize=15, pad=15, color="#1a365d", weight="bold")
plt.show()

In [ ]:
# Geographical Scatter Plot showing house prices
plt.figure(figsize=(10, 8))
plt.scatter(
    df["longitude"], df["latitude"], 
    c=df["median_house_value"], 
    cmap="coolwarm", 
    alpha=0.4, 
    s=df["population"]/100
)
plt.colorbar(label="Median House Value ($)")
plt.title("Geographical Price Distribution", fontsize=15, color="#1a365d", weight="bold")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## 4. Data Preprocessing & Pipeline Construction
To prepare the dataset for Linear Regression, we perform:
1. **Imputation**: Handle missing values in `total_bedrooms` using median imputation.
2. **Scaling**: Standardize numerical variables using `StandardScaler` to help Linear Regression coefficients remain interpretable.
3. **Encoding**: One-hot encode the categorical column `ocean_proximity` so the model can process it.

We wrap these steps using scikit-learn's `ColumnTransformer` and build a comprehensive machine learning `Pipeline`.

In [ ]:
# Separate features and target variable
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

# Define feature columns
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Set up preprocessing pipeline
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

# Build complete Pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])
print("Preprocessing pipeline built successfully!")

## 5. Train / Test Split
We split the data into a **train set (80%)** for fitting the model, and a **test set (20%)** for evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size: {X_test.shape[0]} samples")

## 6. Model Training
We fit the pipeline directly on the training features. This performs numerical imputation, scaling, one-hot encoding, and fits the Linear Regression estimator in one combined workflow.

In [ ]:
print("Training Linear Regression model...")
model_pipeline.fit(X_train, y_train)
print("Training complete!")

## 7. Model Evaluation
We evaluate the model on the test dataset using key metrics:
- **Mean Absolute Error (MAE)**: Average magnitude of the errors.
- **Root Mean Squared Error (RMSE)**: Root of the average squared errors (penalizes larger errors).
- **Coefficient of Determination ($R^2$)**: Proportion of the variance in target variable predicted by features.

In [ ]:
# Run prediction
y_pred = model_pipeline.predict(X_test)

# Compute metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE):     ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R² Score:                       {r2:.4f}")

In [ ]:
# Examine Model Coefficients
regressor = model_pipeline.named_steps["regressor"]
encoder = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]

# Re-construct all feature names
ohe_cat_features = []
for i, col in enumerate(cat_features):
    ohe_cat_features.extend([f"{col}_{val}" for val in encoder.categories_[i]])

all_features = num_features + ohe_cat_features
coefficients = regressor.coef_

coef_df = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", key=abs, ascending=False)

print(f"Model Intercept: ${regressor.intercept_:,.2f}")
print("\nFeature Coefficients sorted by magnitude:")
coef_df

## 8. Results Visualization
We visualize the performance of our model using two diagnostic plots:
1. **Actual vs. Predicted Plot**: Shows predictions against actual values. Ideally, points lie close to the diagonal line.
2. **Residual Plot**: Checks for errors' patterns (should be randomly distributed around zero).

In [ ]:
# Plot Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color="#2b6cb0", edgecolors="w", linewidth=0.5)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color="#e53e3e", linestyle="--", linewidth=2, label="Perfect Prediction")

plt.title("Actual vs. Predicted Median House Value", fontsize=15, color="#1a365d", weight="bold")
plt.xlabel("Actual Value ($)")
plt.ylabel("Predicted Value ($)")
plt.legend(frameon=True, facecolor="white")
plt.show()

In [ ]:
# Plot Residuals
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs. Predicted
axes[0].scatter(y_pred, residuals, alpha=0.3, color="#4a5568", edgecolors="w", linewidth=0.5)
axes[0].axhline(y=0, color="#e53e3e", linestyle="--", linewidth=2)
axes[0].set_title("Residuals vs. Predicted Values", fontsize=14, color="#1a365d", weight="bold")
axes[0].set_xlabel("Predicted Value ($)")
axes[0].set_ylabel("Residual ($)")

# Distribution of Residuals
sns.histplot(residuals, kde=True, color="#4a5568", ax=axes[1], bins=50)
axes[1].axvline(x=0, color="#e53e3e", linestyle="--", linewidth=2)
axes[1].set_title("Distribution of Residuals", fontsize=14, color="#1a365d", weight="bold")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 9. Save Model Pickle
We save the entire pipeline object as `model.pkl` to make it easily deployable in our UI backend.

In [ ]:
model_filename = "model.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(model_pipeline, f)
print(f"Pipeline serialized to {model_filename} successfully!")

## 10. Key Insights & Improvement Ideas
- **Baseline Performance**: The Linear Regression model provides a strong baseline with an $R^2$ of ~0.64. However, it's limited by its linear assumption and sensitivity to outliers.
- **Feature Engineering**: Creating engineered ratios like `rooms_per_household`, `population_per_household`, or `bedrooms_per_room` can capture interaction effects and improve model accuracy.
- **Advanced Models**: Linear Regression is prone to underfitting for complex spatial data. Advanced tree-based algorithms like Random Forests, XGBoost, or Gradient Boosting are likely to perform significantly better.